In [1]:
import pandas as pd
import numpy as np

In [2]:
bm_raw = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Consumption till 13th\bm_raw_new.csv")
bss_pca = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Consumption till 13th\bss_pca_raw.csv")
pca_sellers = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Consumption till 13th\PCA for Sellers.csv")
classification = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\classification_mapping.xlsx")
Brand_mapping = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\Brand_mapping.xlsx")


In [3]:
bss_redemption = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Redemption Reports\BSS Redemption.csv")

In [4]:
free_credit = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Redemption Reports\Free-Credit-Data-Report_2026-01-14.csv")

In [5]:
redemption_Consumption = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Redemption Reports\Redemption-Consumption-Data-Report_2026-01-14.csv")

In [6]:
mtd_brand = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Redemption Reports\MTDOverBurnBrand.csv")

In [7]:
mtd_seller = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Redemption Reports\MTDOverBurnSeller.csv")

In [8]:
input_sheet = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\Input sheet.xlsx")

# BM Raw

In [9]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR'],
      dtype='object')

In [10]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [11]:
# bss_pca.columns

In [12]:
#  pca_sellers.columns

In [13]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [14]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

- New Alpha/MP

In [15]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [16]:
bm_raw['New Alpha/MP'].value_counts()

New Alpha/MP
MP       452982
Alpha     42970
IC         2047
Name: count, dtype: int64

- FCFF Alpha/MP

In [17]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [18]:
bm_raw['FCFF Alpha/MP'].value_counts()

FCFF Alpha/MP
MP       454825
Alpha     36818
HL         6356
Name: count, dtype: int64

- New Supercat

In [19]:
brand_map = dict(zip(Brand_mapping['Vertical'], Brand_mapping['New SC']))
sc_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [20]:
# print(bm_raw['New Supercat'].value_counts().to_string())


- Supercategory matching with FCFF's Super category

In [21]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [22]:
# bm_raw['SC match FCFF'].value_counts()

- New BU

In [23]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [24]:
print(bm_raw['New BU'].isnull().sum())

289


- Spend

In [25]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [26]:
bm_raw['Spends'].sum()

np.float64(1159516729.8400002)

- Brand

In [27]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


# BSS PCA

In [28]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'Ad Account ID',
       'Ad Account Name', 'Business Account ID', 'Business Account Name',
       'Team', 'Alpha_MP', 'BMP_Tag', 'Channel', 'lifestyle_supercategory',
       'spend', 'views', 'clicks', 'ppv', 'click_direct_units',
       'click_indirect_units', 'click_direct_revenue',
       'click_indirect_revenue', 'Product', 'Average CPC', 'CTR', 'ROI'],
      dtype='object')

In [29]:
bss_pca = bss_pca.copy()

- New Alpha/MP

In [30]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


- New Supercat

In [31]:
df_class_unique = classification.drop_duplicates(subset=['Wrong Tagging'])
mapping_dict = dict(zip(df_class_unique['Wrong Tagging'], df_class_unique['New Tag']))

conditions = [
    (bss_pca['BU'] == "Grocery"),
    (bss_pca['BU'] == "Mobile"),
    (bss_pca['Supercategory'] == "PetCare")
]
choices = ["Grocery", "Mobile", "Pets"]
supercat = bss_pca['Supercategory'].map(mapping_dict).fillna(bss_pca['Supercategory'])

bss_pca['New Supercat'] = np.select(conditions, choices, default=supercat)


In [32]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

- Supercategory matching with FCFF's Super category

In [33]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [34]:
bss_pca['SC match FCFF'].value_counts()

SC match FCFF
True     318830
False      5826
Name: count, dtype: int64

- New BU

In [35]:
sc_bu_map = classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

- Brand

In [36]:
brand_map = Brand_mapping.drop_duplicates('Incorrect Account Name').set_index('Incorrect Account Name')['Correct Name'].to_dict()

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

- FCFF Alpha/MP

In [37]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [38]:
bss_pca['BrandxBU'] = bss_pca['Brand'].astype(str) + bss_pca['BU'].astype(str)
lookup_map = bss_pca.drop_duplicates('BrandxBU').set_index('BrandxBU')['New Supercat'].to_dict()

bss_pca['Check'] = bss_pca['BrandxBU'].map(lookup_map)

In [39]:
bss_pca[bss_pca['New BU'] == 0 ]['spend'].sum()

np.float64(0.0)

- check and redo the Supercat and BU Mapping

In [40]:
bss_pca['New Supercat'] = np.where(bss_pca['SC match FCFF'] == False,np.where(bss_pca['Check'] == "Not Assigned", "Not Assigned",bss_pca['Check']),bss_pca['New Supercat'])

In [41]:
sc_bu_map = classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [42]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [43]:
bss_pca[bss_pca['SC match FCFF'] == False]['spend'].sum()

np.float64(453.58)

# PCA for Sellers

In [44]:
pca_sellers.columns

Index(['campaign_id', 'seller_id', 'BU', 'Supercategory', 'Store Name',
       'analytic_super_category', 'lifestyle_supercategory', 'brand',
       'KAM/NKAM', 'BMP_Tag', 'Channel', 'status', 'adspends', 'views',
       'clicks', 'revenue', 'units', 'CTR', 'CVR', 'ROI'],
      dtype='object')

In [45]:
pca_seller = pca_sellers.copy()

- Alpha/MP

In [46]:
pca_seller['Alpha/MP'] = "MP"

- Campaign Type

In [47]:
pca_seller['Campaign Type'] = "PCA"

- New Supercatagory

In [48]:
sc_map_cls = classification.drop_duplicates('Wrong Tagging').set_index('Wrong Tagging')['New Tag'].to_dict()

pca_seller['New Supercatagory'] = pca_seller['Supercategory'].map(sc_map_cls).fillna(pca_seller['Supercategory'])

- Supercategory matching with FCFF's Super category

In [49]:
pca_seller['SC match FCFF'] = pca_seller['New Supercatagory'].isin(classification['Super Categories_Tag'])

In [50]:
pca_seller['SC match FCFF'].value_counts()

SC match FCFF
True     298156
False      7707
Name: count, dtype: int64

In [51]:
print(round(pca_seller[pca_seller['SC match FCFF'] == True]['adspends'].sum()))

9361205


- BU

In [52]:
sc_bu_map =classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()
pca_seller['BU'] = pca_seller['New Supercatagory'].map(sc_bu_map)

In [53]:
#  pca_seller['BU'].value_counts()

In [54]:
brand_sc_map =pca_seller.drop_duplicates('brand').set_index('brand')['Supercategory'].to_dict()
pca_seller['Check'] = pca_seller['brand'].map(brand_sc_map)

- Check and redo Supercat and BU mapping

In [55]:
pca_seller['New Supercatagory'] = np.where(
    (pca_seller['SC match FCFF'] == False) & (pca_seller['New Supercatagory'] != 'Not assigned'),
    pca_seller['Check'],
    pca_seller['New Supercatagory']
)


In [56]:
sc_bu_map =classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()
pca_seller['BU'] = pca_seller['New Supercatagory'].map(sc_bu_map)

In [57]:
pca_seller['SC match FCFF'] = pca_seller['New Supercatagory'].isin(classification['Super Categories_Tag'])

In [58]:
print(round(pca_seller[pca_seller['SC match FCFF'] == True]['adspends'].sum()))

9457362


# MTD Overburn Brand and Sellers

In [59]:
brand_mtd = mtd_brand[["CampaignId","BudgetType","CampaignType","Overburn"]].copy()

In [60]:
seller_mtd = mtd_seller[["CampaignId","BudgetType","CampaignType","Overburn"]].copy()

In [61]:
mtd_overburn = pd.concat([brand_mtd,seller_mtd],axis=0)


In [62]:
print(round(mtd_overburn['Overburn'].sum()))

1746457


In [63]:
mtd_overburn.columns

Index(['CampaignId', 'BudgetType', 'CampaignType', 'Overburn'], dtype='object')

# BSS Rredemption

In [64]:
bss_redemption.columns

Index(['campaign_id', 'upi_id', 'billing_period_hash', 'payment_type',
       'account_id', 'campaign_spend', 'campaign_redeem_fc', 'campaign_redeem',
       'campaign_name', 'campaign_start_date', 'campaign_end_date',
       'campaign_budget', 'campaign_type', 'budget_type', 'market_place',
       'cost_model', 'brands', 'super_category_meta'],
      dtype='object')

- Filtered BSS Redemption Report

In [65]:
bss_redem = bss_redemption[bss_redemption['billing_period_hash'].astype(str).str.startswith('2026-01')].copy()


In [66]:
round(bss_redem['campaign_spend'].sum())

668919999

In [67]:
round(bss_redem['campaign_redeem_fc'].sum())

85172020

In [68]:
round(bss_redem['campaign_redeem'].sum())

583747979

In [69]:
round(bss_redem.groupby("campaign_type")['campaign_redeem'].sum())

campaign_type
BRAND_DISPLAY_ADS        65177.0
BRAND_PCA             72004791.0
BRAND_PLA            511678010.0
Name: campaign_redeem, dtype: float64

# Seller Portal Data

- Redemption&Consumption and FreeCredit Seller Data

In [70]:
free_credit.columns

Index(['campaign_id', 'seller_id', 'amount', 'date', 'merchant_id',
       'compartment_type'],
      dtype='object')

In [71]:
redemption_Consumption.columns

Index(['campaign_id', 'seller_id', 'amount', 'date', 'merchant_id'], dtype='object')

In [72]:
fc = free_credit.copy()
rc = redemption_Consumption.copy()

- Subtracted FC amount from RC amount to get Net FC

In [73]:
rc_unique = rc.groupby(['campaign_id', 'seller_id'])['amount'].sum().reset_index()
fc_unique = fc.groupby(['campaign_id', 'seller_id'])['amount'].sum().reset_index()

df_final = rc_unique.merge(fc_unique, on=['campaign_id', 'seller_id'], how='outer', suffixes=('_rc', '_fc'))

df_final['amount_rc'] = df_final['amount_rc'].fillna(0)
df_final['amount_fc'] = df_final['amount_fc'].fillna(0)

df_final['Net_amount'] = df_final['amount_rc'] - df_final['amount_fc']

rc_final = df_final[['campaign_id', 'seller_id', 'amount_rc', 'amount_fc', 'Net_amount']].copy()


In [74]:
round(rc_final['amount_rc'].sum())

413402776

In [75]:
round(rc_final['amount_fc'].sum())

71398423

In [76]:
round(rc_final['Net_amount'].sum())

342004353

In [77]:
rc_final.columns


Index(['campaign_id', 'seller_id', 'amount_rc', 'amount_fc', 'Net_amount'], dtype='object')

# Search Allocation

1. BM RAW


- BM Raw matched with FCFF Mapping

In [78]:
bm_raw_df = bm_raw[bm_raw['SC match FCFF'] == True][[ 'campaign_id', 'seller_id','FCFF Alpha/MP','New BU',
                    'analytic_vertical','campaign_type', 'Brand', 'Spends',
                     'Actual_spend','New Supercat','Team', 'BMP_Tag','advertiser_id' ]]

In [79]:
bm_raw_df = bm_raw_df.rename(columns={'FCFF Alpha/MP':'Alpha/MP','Team':'KAM/NKAM'})

In [80]:
bm_raw_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'New BU', 'analytic_vertical',
       'campaign_type', 'Brand', 'Spends', 'Actual_spend', 'New Supercat',
       'KAM/NKAM', 'BMP_Tag', 'advertiser_id'],
      dtype='object')

- PCA Sellers matched with FCFF Mapping

In [81]:
pca_seller_df = pca_seller[pca_seller['SC match FCFF'] == True] \
                          [['campaign_id', 'seller_id','Alpha/MP', 'BU','Campaign Type',
                          'brand','adspends', 'New Supercatagory', 'KAM/NKAM','BMP_Tag' ]]

In [82]:
pca_seller_df['Actual_spend'] = pca_seller_df['adspends']

In [83]:
pca_seller_df['analytic_vertical'] = np.nan

In [84]:
pca_seller_df['AdvertiserID'] = np.nan

In [85]:
pca_seller_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'BU', 'Campaign Type', 'brand',
       'adspends', 'New Supercatagory', 'KAM/NKAM', 'BMP_Tag', 'Actual_spend',
       'analytic_vertical', 'AdvertiserID'],
      dtype='object')

In [86]:
pca_seller_df = pca_seller_df.rename(columns={'campaign_id':'campaign_id', 'seller_id':'seller_id', 'Alpha/MP':'Alpha/MP', 'BU':'New BU', 'Campaign Type':'campaign_type', 'brand':'Brand',
       'adspends':'Spends', 'New Supercatagory':'New Supercat', 'KAM/NKAM':'KAM/NKAM', 'BMP_Tag':'BMP_Tag', 'Actual_spend':'Actual_spend',
       'analytic_vertical':'analytic_vertical', 'AdvertiserID':'advertiser_id'})

In [87]:
pca_seller_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'New BU', 'campaign_type',
       'Brand', 'Spends', 'New Supercat', 'KAM/NKAM', 'BMP_Tag',
       'Actual_spend', 'analytic_vertical', 'advertiser_id'],
      dtype='object')

- concatenate bm_raw and pca_seller files to get "Final BM Raw"

In [88]:
bm_raw_final_df = pd.concat([bm_raw_df,pca_seller_df],axis=0)

In [89]:
bm_raw_final_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'New BU', 'analytic_vertical',
       'campaign_type', 'Brand', 'Spends', 'Actual_spend', 'New Supercat',
       'KAM/NKAM', 'BMP_Tag', 'advertiser_id'],
      dtype='object')

In [90]:
# bm_raw_final_df['advertiser_id'].value_counts()

In [91]:
round(bm_raw_final_df['Spends'].sum())

1168837904

- Campaign Type

In [92]:
bm_raw_final_df['Type'] = np.where(bm_raw_final_df['campaign_type'] == "PCA","PCA",bm_raw_final_df['campaign_type'])

In [93]:
bm_raw_final_df['campaign_type'] = np.where(bm_raw_final_df['Type'] == "PCA","PLA",bm_raw_final_df['campaign_type'])

- Shopsy

In [94]:
bm_raw_final_df['Shopsy'] = np.where(bm_raw_final_df['New Supercat'].isin(["ShopsyLifeStyle","ShopsyBGM","ShopsyHome","ShopsyElectronics","ShopsyLarge","ShopsyFurniture"]),"Shopsy","Flipkart")

- Seller Portal Redemption

In [95]:
SP_sums = rc_final.groupby(['campaign_id', 'seller_id'])['Net_amount'].sum()

bm_raw_final_df['mapped_SP_Net'] = (
    bm_raw_final_df.set_index(['campaign_id', 'seller_id'])
    .index.map(SP_sums)
    .fillna(0)
)

denominator = bm_raw_final_df.groupby(['campaign_id', 'seller_id', 'campaign_type'])['Spends'].transform('sum')

bm_raw_final_df['Seller Portal Redemption'] = np.where(
    bm_raw_final_df['campaign_type'] == "PLA",
    (bm_raw_final_df['mapped_SP_Net'] * bm_raw_final_df['Spends']) / denominator,
    0
)

bm_raw_final_df['Seller Portal Redemption'] = bm_raw_final_df['Seller Portal Redemption'].replace([np.inf, -np.inf], 0).fillna(0)


In [96]:
bm_raw_final_df['Seller Portal Redemption'].sum()

np.float64(341928160.46169996)

- converted campaignid and accountid to uppercase to match between bss redemption and bm raw final dataframes

In [97]:
bss_redem['campaign_id'] = bss_redem['campaign_id'].astype(str).str.upper().str.strip()
bss_redem['account_id'] = bss_redem['account_id'].astype(str).str.upper().str.strip()

bm_raw_final_df['campaign_id'] = bm_raw_final_df['campaign_id'].astype(str).str.upper().str.strip()
bm_raw_final_df['advertiser_id'] = bm_raw_final_df['advertiser_id'].astype(str).str.upper().str.strip()


- BSS Redemption

In [98]:
bss_totals = bss_redem.groupby(['campaign_id', 'campaign_type', 'account_id'])['campaign_redeem'].sum()
bss_totals.index.names = ['campaign_id', 'campaign_type', 'advertiser_id']

bm_raw_map = bm_raw_final_df.set_index(['campaign_id', 'campaign_type', 'advertiser_id']).index
bm_raw_final_df['BSS_Campaign_redeem'] = bm_raw_map.map(bss_totals).fillna(0)


denominator = bm_raw_final_df.groupby(['campaign_id', 'campaign_type', 'advertiser_id'])['Actual_spend'].transform('sum')

bm_raw_final_df['BSS Redemption'] = np.where(
    (bm_raw_final_df['campaign_type'] == "BRAND_PLA") & (denominator > 0),
    (bm_raw_final_df['BSS_Campaign_redeem'] * bm_raw_final_df['Actual_spend']) / denominator,
    0
)


In [99]:
round(bm_raw_final_df['BSS Redemption'].sum())

511674089

- Free Credit for Brand_PLA

In [100]:
bss_totals = bss_redem.groupby(['campaign_id', 'campaign_type', 'account_id'])['campaign_redeem_fc'].sum()
bss_totals.index.names = ['campaign_id', 'campaign_type', 'advertiser_id']

bm_raw_map = bm_raw_final_df.set_index(['campaign_id', 'campaign_type', 'advertiser_id']).index
bm_raw_final_df['BSS_Campaign_redeem'] = bm_raw_map.map(bss_totals).fillna(0)


denominator = bm_raw_final_df.groupby(['campaign_id', 'campaign_type', 'advertiser_id'])['Spends'].transform('sum')

bm_raw_final_df['Free Credit BPLA'] = np.where(
    (bm_raw_final_df['campaign_type'] == "BRAND_PLA") & (denominator > 0),
    (bm_raw_final_df['BSS_Campaign_redeem'] * bm_raw_final_df['Spends']) / denominator,
    0
)


In [101]:
round(bm_raw_final_df['Free Credit BPLA'].sum())

71612008

- Free Credit for PLA

In [102]:
SP_sums = rc_final.groupby(['campaign_id', 'seller_id'])['amount_fc'].sum()

bm_raw_final_df['mapped_SP_Net'] = (
    bm_raw_final_df.set_index(['campaign_id', 'seller_id'])
    .index.map(SP_sums)
    .fillna(0)
)

denominator = bm_raw_final_df.groupby(['campaign_id', 'seller_id', 'campaign_type'])['Spends'].transform('sum')

bm_raw_final_df['Free Credit PLA'] = np.where(
    bm_raw_final_df['campaign_type'] == "PLA",
    (bm_raw_final_df['mapped_SP_Net'] * bm_raw_final_df['Spends']) / denominator,
    0
)

bm_raw_final_df['Free Credit PLA'] = bm_raw_final_df['Free Credit PLA'].replace([np.inf, -np.inf], 0).fillna(0)


In [103]:
round(bm_raw_final_df['Free Credit PLA'].sum())

61373583

converted campaignid and campaigntype to uppercase to get exact match

In [104]:
mtd_overburn['CampaignId'] = mtd_overburn['CampaignId'].astype(str).str.upper().str.strip()
mtd_overburn['CampaignType'] = mtd_overburn['CampaignType'].astype(str).str.upper().str.strip()

- OverBurn for Brand_PLA

In [105]:
overburn_totals = mtd_overburn.groupby(['CampaignId', 'CampaignType'])['Overburn'].sum()

bm_raw_final_df['mapped_Overburn'] = (
    bm_raw_final_df.set_index(['campaign_id', 'campaign_type'])
    .index.map(overburn_totals)
    .fillna(0)
)

denominator = bm_raw_final_df.groupby(['campaign_id', 'campaign_type'])['Spends'].transform('sum')

bm_raw_final_df['Overburn_BPLA'] = np.where(
    (bm_raw_final_df['campaign_type'] == "BRAND_PLA") & (denominator > 0),
    (bm_raw_final_df['mapped_Overburn'] * bm_raw_final_df['Spends']) / denominator,
    0
)


In [106]:
bm_raw_final_df['Overburn_BPLA'].sum()

np.float64(702466.24)

- OverBurn for PLA

In [107]:
overburn_totals = mtd_overburn.groupby(['CampaignId', 'CampaignType'])['Overburn'].sum()

bm_raw_final_df['mapped_Overburn'] = (
    bm_raw_final_df.set_index(['campaign_id', 'campaign_type'])
    .index.map(overburn_totals)
    .fillna(0)
)

denominator = bm_raw_final_df.groupby(['campaign_id', 'campaign_type'])['Spends'].transform('sum')

bm_raw_final_df['Overburn_PLA'] = np.where(
    (bm_raw_final_df['campaign_type'] == "PLA") & (denominator > 0),
    (bm_raw_final_df['mapped_Overburn'] * bm_raw_final_df['Spends']) / denominator,
    0
)


In [108]:
round(bm_raw_final_df['Overburn_PLA'].sum())

1043645

- Net Spends

In [109]:
bm_raw_final_df['Net Spend'] = bm_raw_final_df['Spends'] - bm_raw_final_df['Free Credit BPLA'] - bm_raw_final_df['Free Credit PLA'] - bm_raw_final_df['Overburn_BPLA'] - bm_raw_final_df['Overburn_PLA']

In [110]:
round(bm_raw_final_df['Net Spend'].sum())

1034106201

- Final Spends

In [111]:
bm_raw_final_df['Final_Spends'] = bm_raw_final_df['BSS Redemption'] + bm_raw_final_df['Seller Portal Redemption']

In [112]:
round(bm_raw_final_df['Final_Spends'].sum())

853602249

- Market FC, Alpha FC- MP Spillover, MP FC- MP Spillover are zero's because there is base to bifurcate them

In [113]:
bm_raw_final_df['Market FC'] = np.nan

In [114]:
bm_raw_final_df['Alpha FC- MP Spillover'] = np.nan

In [115]:
bm_raw_final_df['MP FC- MP Spillover'] = np.nan

- Final (Amount)

In [116]:
bm_raw_final_df['Final'] = bm_raw_final_df['Final_Spends']

In [117]:
round(bm_raw_final_df['Final'].sum())

853602249

- Estimates


In [118]:
mul_fac_pla = 0.85
mul_fac_pca = 1.0
remaindays_sp = 20
remaindays_bss = 20
consumeddays_sp = 11
consumeddays_bss = 11

In [119]:
bm_raw_final_df['Estimates'] = np.where(
    bm_raw_final_df['campaign_type'] == "PLA",
    ((bm_raw_final_df['Final'] / consumeddays_sp) * mul_fac_pla * remaindays_sp), 
    ((bm_raw_final_df['Final'] / consumeddays_bss) * mul_fac_pca * remaindays_bss)  
) + bm_raw_final_df['Final']

In [120]:
print(f'final estimates is {bm_raw_final_df['Estimates'].sum()}')

final estimates is 2312353204.6079636


2. PCA Consumption

In [121]:
bss_pca_df = bss_pca[bss_pca['SC match FCFF'] == True].copy()

In [122]:
bss_pca_final_df = bss_pca_df[['Campaign ID','Ad Account ID', 'BU',
                            'New Supercat','FCFF Alpha/MP', 'Brand',
                            'Team', 'Store Name','spend']]

In [123]:
bss_pca_final_df = bss_pca_final_df.rename(columns={"Campaign ID":"Campaign ID",
                                              "Ad Account ID":"Ad Account ID",
                                              "BU":"BU","New Supercat":"Super category",
                                              "FCFF Alpha/MP":"Alpha/MP",
                                              "Brand":"Brand","Team":"KAM/NKAM",
                                              "Store Name":"Store Name","spend":"Spend"})

- Campaign Type

In [124]:
bss_pca_final_df['Campaign Type'] = "BRAND_PCA"

In [125]:
bss_pca_final_df.columns

Index(['Campaign ID', 'Ad Account ID', 'BU', 'Super category', 'Alpha/MP',
       'Brand', 'KAM/NKAM', 'Store Name', 'Spend', 'Campaign Type'],
      dtype='object')

- Free Credit for Brand_PCA

In [126]:
bss_totals = bss_redem.groupby(['campaign_id', 'campaign_type', 'account_id'])['campaign_redeem_fc'].sum()
bss_totals.index.names = ['campaign_id', 'campaign_type', 'advertiser_id']

bss_pca_map = bss_pca_final_df.set_index(['Campaign ID', 'Campaign Type', 'Ad Account ID']).index
bss_pca_final_df['BSS_Campaign_redeem'] = bss_pca_map.map(bss_totals).fillna(0)


denominator = bss_pca_final_df.groupby(['Campaign ID', 'Campaign Type', 'Ad Account ID'])['Spend'].transform('sum')

bss_pca_final_df['Free Credit BPCA'] = np.where(
    (bss_pca_final_df['Campaign Type'] == "BRAND_PCA") & (denominator > 0),
    (bss_pca_final_df['BSS_Campaign_redeem'] * bss_pca_final_df['Spend']) / denominator,
    0
)


In [127]:
round(bss_pca_final_df['Free Credit BPCA'].sum())

13471688

- OverBurn for Brand_PCA

In [128]:
overburn_totals = mtd_overburn.groupby(['CampaignId', 'CampaignType'])['Overburn'].sum()

bss_pca_final_df['mapped_Overburn'] = (
    bss_pca_final_df.set_index(['Campaign ID', 'Campaign Type'])
    .index.map(overburn_totals)
    .fillna(0)
)

denominator = bss_pca_final_df.groupby(['Campaign ID', 'Campaign Type'])['Spend'].transform('sum')

bss_pca_final_df['Overburn_BPLA'] = np.where(
    (bss_pca_final_df['Campaign ID'] == "BRAND_PCA") & (denominator > 0),
    (bss_pca_final_df['mapped_Overburn'] * bss_pca_final_df['Spend']) / denominator,
    0
)


In [129]:
round(bss_pca_final_df['Overburn_BPLA'].sum())

0

- Marketing FC

In [130]:
bss_pca_final_df['Marketing FC'] = np.nan

- PCA Redemption

In [131]:
bss_totals = bss_redem.groupby(['campaign_id', 'campaign_type', 'account_id'])['campaign_redeem'].sum()
bss_totals.index.names = ['campaign_id', 'campaign_type', 'advertiser_id']

bss_pca_map = bss_pca_final_df.set_index(['Campaign ID', 'Campaign Type', 'Ad Account ID']).index
bss_pca_final_df['BSS_Campaign_redeem'] = bss_pca_map.map(bss_totals).fillna(0)

denominator = bss_pca_final_df.groupby(['Campaign ID', 'Campaign Type', 'Ad Account ID'])['Spend'].transform('sum')

bss_pca_final_df['PCA Redemption'] = np.where(
    (bss_pca_final_df['Campaign Type'] == "BRAND_PCA") & (denominator > 0),
    (bss_pca_final_df['BSS_Campaign_redeem'] * bss_pca_final_df['Spend']) / denominator,
    0
)


In [132]:
round(bss_pca_final_df['PCA Redemption'].sum())

72003268

- Net Spend

In [133]:
bss_pca_final_df['Net Spend'] = bss_pca_final_df['Spend'] - bss_pca_final_df['Free Credit BPCA'] - bss_pca_final_df['Overburn_BPLA']

In [134]:
round(bss_pca_final_df['Net Spend'].sum())

85145871

- Final Spend

In [135]:
bss_pca_final_df['Final Spend'] = bss_pca_final_df['PCA Redemption']

In [136]:
round(bss_pca_final_df['Final Spend'].sum())

72003268

In [137]:

bss_pca_final_df['Estimates'] = ((bss_pca_final_df['Final Spend'] / consumeddays_bss) * remaindays_bss * mul_fac_pca) + bss_pca_final_df['Final Spend']


In [138]:
bss_pca_final_df['Estimates'].sum()

np.float64(202918299.65636367)

In [139]:
input_sheet.columns

Index(['Category - BM Raw', 'Supercat Tagging'], dtype='object')

- New Supercategory

In [140]:
map_cat2sc = dict(zip(input_sheet['Category - BM Raw'], input_sheet['Supercat Tagging']))

map_sc2sc = dict(zip(input_sheet['Supercat Tagging'], input_sheet['Supercat Tagging']))

conditions = [
    (bss_pca_final_df['BU'] == 'BGM') & (bss_pca_final_df['Super category'] == 'PersonalHealthCare')
]

lookup_result = bss_pca_final_df['Super category'].map(map_cat2sc).fillna(bss_pca_final_df['Super category'].map(map_sc2sc)).fillna(bss_pca_final_df['Super category'])

choices = ["Grooming"]

bss_pca_final_df['New Super_Category'] = np.select(conditions, choices, default=lookup_result)
